# Lesson 6: Structured Data

Port of **Structured Data** (Claude course: Lesson 8) to the Gemini API.

The problem is the same on every model: ask for JSON and you get **markdown fences plus a friendly explanation**, so a user can't just copy the output.

The Claude lesson fixes this with **assistant prefill + stop sequences**. Gemini supports stop sequences but **rejects a trailing assistant turn**, so this notebook shows that failure directly and then uses Gemini's native JSON mode, which is the intended solution:

1. Naive request → dirty output.
2. Prefill attempt → `400 INVALID_ARGUMENT`.
3. Stop sequence alone → truncates *before* the JSON.
4. `response_mime_type="application/json"` → clean, parseable JSON.
5. `response_schema` → guaranteed shape.

In [5]:
import json
import time

from dotenv import load_dotenv
from google import genai
from google.genai import types
from google.genai.errors import ClientError, ServerError

load_dotenv(override=True)

client = genai.Client()
model = "gemini-3.6-flash"

In [6]:
def add_user_message(messages, text):
    messages.append({"role": "user", "parts": [{"text": text}]})


def add_assistant_message(messages, text):
    # Gemini calls the assistant role "model"
    messages.append({"role": "model", "parts": [{"text": text}]})


def generate_with_retry(**params):
    for attempt in range(5):
        try:
            return client.models.generate_content(**params)
        except ServerError:
            if attempt == 4:
                raise
            time.sleep(2 ** attempt)


def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    config_kwargs = {
        "max_output_tokens": 1000,
        "temperature": temperature,
    }
    if system:
        config_kwargs["system_instruction"] = system
    if stop_sequences:
        config_kwargs["stop_sequences"] = stop_sequences

    params = {
        "model": model,
        "contents": messages,
        "config": types.GenerateContentConfig(**config_kwargs),
    }
    return generate_with_retry(**params).text

## 1. The problem: default output is not clean

No formatting instructions — just ask for JSON. You get commentary plus a `````json` fence.

In [7]:
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")

naive = chat(messages)
print(naive)

print("\ncontains a code fence?", "```" in (naive or ""))
print("parses as JSON?", end=" ")
try:
    json.loads((naive or "").strip())
    print(True)
except json.JSONDecodeError:
    print(False)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 17h41m5.200678793s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per day limit exceeded on Free Tier. See htt

## 2. The Claude trick: prefill + stop sequence

Claude lets the conversation end on an assistant turn:

```python
add_assistant_message(messages, "```json")
text = chat(messages, stop_sequences=["```"])
```

Gemini does not. The model must be the one to start its turn, so a request whose last turn is `role: "model"` is rejected outright. The next cell reproduces the error on purpose.

In [ ]:
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

try:
    text = chat(messages, stop_sequences=["```"])
    print("Unexpected success:", repr(text))
except ClientError as exc:
    print("Gemini rejected the prefill:")
    print(exc)

### Stop sequences alone don't fix it

`config.stop_sequences` works in Gemini, but a stop sequence can only *cut* text — it cannot supply the missing opening fence.

So generation runs until the fence the model writes at the *start* of its JSON block, and everything after it is discarded. Depending on the run you get only the commentary, or `None` if the model opened with a fence immediately.

In [ ]:
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")

truncated = chat(messages, stop_sequences=["```"])
print(repr(truncated))

print("\nparses as JSON?", end=" ")
try:
    json.loads((truncated or "").strip())
    print(True)
except json.JSONDecodeError:
    print(False)

## 3. The Gemini fix: JSON mode

Set `response_mime_type="application/json"` and the model emits **only valid JSON** — no fence, no commentary, no prefill needed.

This is the production-safe replacement for the Claude prefill trick.

In [ ]:
response = generate_with_retry(
    model=model,
    contents="Generate a very short event bridge rule as json",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        max_output_tokens=1000,
    ),
)

print(response.text)
print("\nparsed:", json.loads(response.text))

### Enforcing the shape with `response_schema`

JSON mode guarantees valid JSON, not the keys your app expects. Pass a Pydantic model as `response_schema` and Gemini fills that exact structure; `response.parsed` hands back a real object instead of a string.

Note: Python identifiers can't contain hyphens, so `detail_type` stands in for the EventBridge key `detail-type`. Rename after parsing if the consumer needs the literal key.

In [ ]:
from pydantic import BaseModel


class RuleDetail(BaseModel):
    state: list[str]


class EventBridgeRule(BaseModel):
    source: list[str]
    detail_type: list[str]
    detail: RuleDetail


response = generate_with_retry(
    model=model,
    contents="Generate a very short event bridge rule as json.",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=EventBridgeRule,
        max_output_tokens=1000,
    ),
)

rule = response.parsed
print(type(rule).__name__)
print(rule.model_dump_json(indent=2))
print("\nfirst source:", rule.source[0])

## 4. Beyond JSON: Python code

JSON mode only covers JSON. For code there is no prefill on Gemini, so the equivalent is a strict system instruction — the same fallback you'd use for CSV, SQL, or a bare bulleted list.

It is a request, not a guarantee: if the model still adds fences, strip them before use.

In [ ]:
messages = []
add_user_message(messages, "Write a very short Python function that adds two numbers.")

code = chat(
    messages,
    system="Return only raw code. No markdown fences, no explanation.",
)
print(code)

## Notes for the Gemini port

| Claude | Gemini |
|---|---|
| `add_assistant_message(messages, "```json")` prefill | **Not supported** — `400 INVALID_ARGUMENT: Requests ending with a model turn are not supported.` |
| `stop_sequences=["```"]` | `config.stop_sequences=["```"]` (works, but cannot supply the opening fence) |
| — | `config.response_mime_type="application/json"` — the actual fix |
| — | `config.response_schema=MyModel` + `response.parsed` — typed output |

- **Prefill rejection is model/API-version specific.** `gemini-3.6-flash` is the model used across this repo; older Gemini models accepted prefills, so don't be surprised if a tutorial shows the Claude pattern working.
- **`response.parsed` is only populated when `response_schema` is set.** Without a schema you still have to `json.loads` the text yourself.
- **Stop sequences truncate silently.** The response comes back with `finish_reason=STOP` and no error, so validate the payload instead of assuming it is complete.
- **`max_output_tokens`** is Claude's `max_tokens`; too low truncates the JSON and breaks `json.loads`.